In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import numpy as np
import os

In [7]:
base_dir = os.getcwd()
data_folder = os.path.join(base_dir, "data")

In [9]:
data_2022= pd.read_csv(os.path.join(data_folder, 'processed_22.csv'))
data_2023 = pd.read_csv(os.path.join(data_folder, 'processed_23.csv'))
data_2024 = pd.read_csv(os.path.join(data_folder, 'processed_24.csv'))
data_2025 = pd.read_csv(os.path.join(data_folder, 'processed_25.csv'))
total = pd.concat([data_2022, data_2023, data_2024, data_2025], ignore_index=True)  

In [ ]:
total["date"] = pd.to_datetime(total["date"])
y = total['Lpoly_expected_ml']
X = total.drop(columns=["date", "Lpoly_expected", "Lpoly_expected_ml","Pmicans_expected","Pmicans_expected_ml"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=23)

In [69]:
GB = GradientBoostingRegressor(learning_rate=0.1, n_estimators=100, subsample=0.8, max_depth=2, min_samples_leaf=5 ,max_features=0.5, random_state=23)
GB.fit(X_train, y_train)
y_pred_gb_train = GB.predict(X_train) 
mse_gb_train = mean_squared_error(y_train, y_pred_gb_train)
r2_gb_train = r2_score(y_train, y_pred_gb_train)
print(f"Gradient Boosting Mean Squared Error: {mse_gb_train:.2f}")
print(f"Gradient Boosting R^2 Score: {r2_gb_train:.2f}")


Gradient Boosting Mean Squared Error: 6.66
Gradient Boosting R^2 Score: 0.92


In [70]:
y_pred_gb_test = GB.predict(X_test)
mse_gb_test = mean_squared_error(y_test, y_pred_gb_test)
r2_gb_test = r2_score(y_test, y_pred_gb_test)   
print(f"Gradient Boosting Mean Squared Error: {mse_gb_test:.2f}")
print(f"Gradient Boosting R^2 Score: {r2_gb_test:.2f}")

Gradient Boosting Mean Squared Error: 60.04
Gradient Boosting R^2 Score: 0.60


In [73]:
feature_importance_df = pd.DataFrame({'Feature' : X_train.columns, 'Importance': GB.feature_importances_})
feature_importance_df.drop(feature_importance_df[feature_importance_df['Importance'] == 0].index, inplace=True)
feature_importance_df.head(20).sort_values(by='Importance',ascending=False)

,Feature,Importance
2,ROI_per_ml,0.113927
32,moment_invariant2,0.107171
31,moment_invariant1,0.099732
1,ml_analyzed,0.087414
0,roiCount,0.073845
39,shapehist_kurtosis_normEqD,0.054207
15,H180,0.037722
33,moment_invariant3,0.027286
12,Eccentricity,0.023152
16,H90,0.012767


In [79]:
gb_params = {
    'learning_rate':    [0.01, 0.05, 0.1],
    'n_estimators':     [100, 200],
    'subsample':        [0.6, 0.7, 0.8],
    'max_depth':        [2, 3],
    'min_samples_leaf': [5, 10, 15],
    
}
base_GBR = GradientBoostingRegressor(random_state=23)
gb_grid = GridSearchCV(base_GBR,gb_params,cv=5,n_jobs=-1,verbose=1,scoring='r2')
gb_grid.fit(X_train, y_train)
print(f"Best Hyperparameters: {gb_grid.best_params_}")
best_gb = gb_grid.best_estimator_



Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 10, 'n_estimators': 200, 'subsample': 0.8}


In [ ]:
y_best_pred_train = best_gb.predict(X_train)
mse_train_best = mean_squared_error(y_train, y_best_pred_train)
r2_train_best = r2_score(y_train, y_best_pred_train)
print(f"Best Gradient Boosting Mean Squared Error: {mse_train_best:.2f}")
print(f"Best Gradient Boosting R^2 Score: {r2_train_best:.2f}")

Best Gradient Boosting Mean Squared Error: 5.75
Best Gradient Boosting R^2 Score: 0.93


In [84]:
y_best_pred_test = best_gb.predict(X_test)
mse_test_best = mean_squared_error(y_test, y_best_pred_test)
r2_test_best = r2_score(y_test, y_best_pred_test)
print(f"Best Gradient Boosting Test Mean Squared Error: {mse_test_best:.2f}")
print(f"Best Gradient Boosting Test R^2 Score: {r2_test_best:.2f}")

Best Gradient Boosting Test Mean Squared Error: 53.37
Best Gradient Boosting Test R^2 Score: 0.64
